# PILOT — CE-crossing selector (gold-free) tren Forget-MI  [INLINE, an toan Save Version]

Kiem tra 4 cach chon checkpoint KHONG dung GOLD/final-test, chay **NGAY TRONG luc train**
(khong can luu 30 checkpoint -> khong loi dia/path):
- **S1** First Crossing · **S2** Closest CE · **S3** First Stable Crossing · **S4** CE-match + Utility.

Luong: (Cell 3) chay Forget-MI 1 luot voi `ce_selector_out` -> moi epoch tinh CE (D_f vs D_nm_val)
+ snapshot 4 ung vien; cuoi cung eval selected tren D_t_final. (Cell 4-5) xem ket qua.
**KHONG doi training/loss Forget-MI** (chi eval read-only + snapshot, RNG-safe).


In [ ]:
# Cell 1: setup
import os, subprocess
WORK='/kaggle/working'; REPO=f'{WORK}/Forget-MI-LoKU'
if not os.path.isdir(REPO):
    subprocess.run(['git','clone','https://github.com/nhnhu146/Forget-MI-LoKU.git',REPO],check=True)
else:
    subprocess.run(['git','-C',REPO,'pull','--ff-only'],check=True)
os.chdir(REPO)
assert os.path.exists('training/ce_selector_pilot.py'),'push code truoc + re-import notebook'
subprocess.run(['pip','install','-q','pydicom','scikit-image','scikit-learn','pyyaml','wandb','seaborn==0.13.2'],check=True)
subprocess.run(['pip','install','-q','transformers==4.38.0','peft==0.10.0','accelerate==0.27.0'],check=True)
import torch; assert torch.cuda.is_available(),'Bat GPU'
print('Commit:',subprocess.check_output(['git','rev-parse','--short','HEAD'],text=True).strip())
print('GPU   :',torch.cuda.get_device_name(0))


In [ ]:
# Cell 2: path discovery (giong baseline)
import glob, os
FORGET_PCT = 3        # 3 | 6 | 10
SEED       = 42
assert FORGET_PCT in (3,6,10)

def find_dataset(*slugs):
    for slug in slugs:
        if os.path.isdir(f'/kaggle/input/{slug}'): return f'/kaggle/input/{slug}'
        hits=glob.glob(f'/kaggle/input/datasets/*/{slug}')
        if hits: return sorted(hits)[0]
    return None
def first_existing(root, rels):
    for r in rels:
        p=os.path.join(root,r)
        if os.path.exists(p): return p
    return None

DATA_ROOT=find_dataset('forget-mi-data')
MODELS_ROOT=find_dataset('forget-mi-models-full','forget-mi-models-v2','forget-mi-models')
assert DATA_ROOT and MODELS_ROOT,'Add forget-mi-data + forget-mi-models-full'
base_hits=glob.glob(os.path.join(MODELS_ROOT,'**','training_original_model','pytorch_model.bin'),recursive=True)
gold_hits=glob.glob(os.path.join(MODELS_ROOT,'**',f'model_retrained_{FORGET_PCT}per','**','pytorch_model.bin'),recursive=True)
BASE_MODEL=os.path.dirname(sorted(base_hits,key=len)[0]) if base_hits else None
GOLD_MODEL=os.path.dirname(sorted(gold_hits,key=len)[0]) if gold_hits else BASE_MODEL
TEXT_DIR=first_existing(DATA_ROOT,['data/metadata','metadata'])
IMG_DIR =first_existing(DATA_ROOT,['data/img_data','img_data'])
FORGET_CSV=f'./data_splits/forget_set_{FORGET_PCT}per.csv'

RUN_ID    =f'forgetmi_pilot_{FORGET_PCT}per_s{SEED}'
OUTPUT_DIR=f'/kaggle/working/pilot_output/{FORGET_PCT}per_s{SEED}'
SEL_DIR   =f'/kaggle/working/checkpoint_selection_{FORGET_PCT}per_s{SEED}'
for n,p in {'base':BASE_MODEL,'text':TEXT_DIR,'img':IMG_DIR,'forget':FORGET_CSV}.items():
    assert p and os.path.exists(p),f'Missing {n}: {p}'

COMMON_OVR={'forget_set_path':FORGET_CSV,'base_model_path':BASE_MODEL,'bert_pretrained_dir':BASE_MODEL,
            'retrained_model_path':GOLD_MODEL,'text_data_dir':TEXT_DIR,'img_data_dir':IMG_DIR}
print('FORGET_PCT',FORGET_PCT,'SEED',SEED); print('BASE',BASE_MODEL); print('SEL_DIR',SEL_DIR)


In [ ]:
# Cell 3: TRAIN Forget-MI + CE-SELECTOR INLINE (1 luot, ~50 phut)
#   ce_selector_out=SEL_DIR -> moi epoch tinh CE (D_f vs D_nm_val) + snapshot 4 ung vien (~2GB),
#     KHONG luu 30 checkpoint (tranh loi dia/path).
#   evaluate_last_and_best=1 -> baseline chi luu last/val_best (~900MB). eval_every_epoch=0.
#   KHONG doi training/loss Forget-MI (selector chi eval read-only + snapshot, RNG-safe).
import os, subprocess, time
ovr=dict(COMMON_OVR); ovr.update({'output_dir':OUTPUT_DIR,
    'results_csv_path':'/kaggle/working/results_pilot_native.csv',
    'evaluate_last_and_best':1, 'eval_every_epoch':0, 'id':RUN_ID,
    'ce_selector_out':SEL_DIR, 's4_delta':0.15})
arg=','.join(f'{k}={v}' for k,v in ovr.items())
env={**os.environ,'PYTHONPATH':'.','WANDB_MODE':'disabled'}
cmd=['python','training/forgetmi_partial.py','--config','config_baseline_kaggle.yaml',
     '--seed',str(SEED),'--fresh','--override',arg]
print('='*70); print('TRAIN + INLINE CE-SELECTOR',RUN_ID); print('='*70)
t0=time.time()
try:
    subprocess.run(cmd,env=env,check=True); print(f'DONE ({(time.time()-t0)/3600:.2f}h)')
except subprocess.CalledProcessError as e:
    print(f'FAIL rc={e.returncode}')


In [ ]:
# Cell 4: kiem tra output selector (da tao INLINE o Cell 3)
import os, glob
print('SEL_DIR:',SEL_DIR)
print('Files  :',sorted(os.path.basename(p) for p in glob.glob(f'{SEL_DIR}/*')))
assert os.path.exists(os.path.join(SEL_DIR,'selected_checkpoints.json')),\
    'Khong thay selected_checkpoints.json — Cell 3 chay xong chua? (xem log Cell 3 co dong 🧭 selector khong)'


In [ ]:
# Cell 5: xem ket qua 4 selector
import os, json, pandas as pd
pd.set_option('display.width',180)
traj=os.path.join(SEL_DIR,'forgetmi_selector.csv')
if os.path.exists(traj):
    print('===== TRAJECTORY (gold-free: forget_ce vs nm_val_ce + utility) =====')
    print(pd.read_csv(traj).to_string(index=False))
sj=os.path.join(SEL_DIR,'selected_checkpoints.json')
if os.path.exists(sj):
    d=json.load(open(sj)); res=d['results']
    print(f"\n===== 4 CACH CHON (s4_delta={d.get('s4_delta')}) =====")
    tab=[]
    for k,v in res.items():
        if v.get('epoch') is None or 'Df_AUC' not in v:
            tab.append({'selector':k,'epoch':(f"E{v.get('epoch')}" if v.get('epoch') is not None else 'NO CROSSING')})
        else:
            tab.append({'selector':k,'epoch':f"E{v['epoch']}",'Df_AUC':v['Df_AUC'],'Df_F1':v['Df_F1'],
                        'Dt_AUC':v['Dt_AUC'],'Dt_F1':v['Dt_F1'],'MIA':v['MIA']})
    print(pd.DataFrame(tab).to_string(index=False))
    uniq=sorted(set(v['epoch'] for v in res.values() if v.get('epoch') is not None))
    print(f"\n-> {len(uniq)} epoch khac nhau: {['E'+str(e) for e in uniq]}  "
          f"({'DONG THUAN cao' if len(uniq)<=2 else 'phan tan'})")
print('\nOutput:',SEL_DIR)
